In [ ]:
%iam_role arn:aws:iam::770170581396:role/aws-glue-s3-permission
%region eu-north-1
%idle_timeout 15
%worker_type G.1X
%number_of_workers 2
%glue_version 5.0

# Silver to Gold

Analyze Silver and publish two transformation-only Gold contracts: binary sentiment model input and sentence-level aspect input. This notebook does not train a model, balance training data, run LDA, apply an aspect lexicon, or score sentiment.

In [ ]:
from pyspark.sql import Window
from pyspark.sql import functions as F

SILVER_PATH = "s3://amazon-food-reviews-ml-model/silver/"
GOLD_MODEL_PATH = "s3://amazon-food-reviews-ml-model/gold/model_input/"
GOLD_ASPECT_PATH = "s3://amazon-food-reviews-ml-model/gold/aspect_sentences/"

silver_df = spark.read.parquet(SILVER_PATH).cache()
silver_count = silver_df.count()
if silver_count == 0:
    raise ValueError(f"No Silver data found at {SILVER_PATH}")

print(f"Silver rows: {silver_count:,}")
silver_df.printSchema()

## 1. Silver Analysis

In [ ]:
required_columns = {
    "id", "product_id", "user_id", "score",
    "sentiment_class", "binary_label", "review_text_clean",
    "normalized_text", "review_length", "review_word_count",
    "review_time_epoch", "review_timestamp",
    "helpfulness_numerator", "helpfulness_denominator",
    "helpfulness_ratio", "_record_id",
}
missing_columns = sorted(required_columns - set(silver_df.columns))
if missing_columns:
    raise ValueError(f"Missing Silver columns: {missing_columns}")

silver_df.groupBy("score", "sentiment_class", "binary_label").count().orderBy("score").show()

silver_df.agg(
    F.countDistinct("product_id").alias("products"),
    F.countDistinct("user_id").alias("users"),
    F.countDistinct("_record_id").alias("record_ids"),
    F.count(F.when(F.col("review_timestamp").isNull(), 1)).alias("missing_timestamps"),
    F.round(F.avg("review_length"), 1).alias("average_review_chars"),
).show(truncate=False)

## 2. Gold Model-Input Transformation

Exclude neutral ratings only here, prevent duplicate-text leakage, and assign deterministic 80/10/10 label-stratified splits. Future class balancing must sample only the training split.

In [ ]:
duplicate_window = Window.partitionBy("normalized_text").orderBy(
    F.col("helpfulness_denominator").desc(), F.col("id"), F.col("_record_id")
)
deduplicated_df = (
    silver_df.filter(F.col("binary_label").isin(0, 1))
    .withColumn("_duplicate_rank", F.row_number().over(duplicate_window))
    .filter(F.col("_duplicate_rank") == 1).drop("_duplicate_rank")
)

split_window = Window.partitionBy("binary_label").orderBy(F.xxhash64("_record_id"), "_record_id")
label_window = Window.partitionBy("binary_label")
model_input_df = (
    deduplicated_df
    .withColumn("_row_number", F.row_number().over(split_window))
    .withColumn("_label_count", F.count(F.lit(1)).over(label_window))
    .withColumn("_fraction", F.col("_row_number") / F.col("_label_count"))
    .withColumn("dataset_split",
        F.when(F.col("_fraction") <= 0.80, "train")
         .when(F.col("_fraction") <= 0.90, "validation").otherwise("test"))
    .select(
        F.col("_record_id").alias("record_id"),
        F.col("normalized_text").alias("clean_text"),
        F.col("binary_label").cast("int").alias("label"),
        "dataset_split", "score", "product_id", "user_id",
        "review_timestamp", "review_length", "review_word_count",
        "helpfulness_numerator", "helpfulness_denominator", "helpfulness_ratio",
    )
)
model_count = model_input_df.count()
print(f"Gold model-input rows: {model_count:,}")
model_input_df.groupBy("dataset_split", "label").count().orderBy("dataset_split", "label").show()

## 3. Gold Aspect-Sentence Transformation

Retain negative, neutral, and positive reviews. Explode punctuation-preserving review text into sentences while carrying product, time, rating, and helpfulness context needed for the later hybrid aspect layer.

In [ ]:
sentence_ready_df = silver_df.withColumn(
    "_sentences", F.split(F.col("review_text_clean"), r"(?<=[.!?])\s+")
)
aspect_sentences_df = (
    sentence_ready_df
    .select(
        "_record_id", "id", "product_id", "user_id", "score",
        "sentiment_class", "review_time_epoch", "review_timestamp",
        "helpfulness_numerator", "helpfulness_denominator", "helpfulness_ratio",
        F.posexplode("_sentences").alias("sentence_index", "sentence_text"),
    )
    .withColumn("sentence_text", F.trim("sentence_text"))
    .filter(F.length("sentence_text") > 0)
    .withColumn("sentence_normalized", F.trim(F.regexp_replace(F.lower("sentence_text"), r"\s+", " ")))
    .withColumn("review_year", F.year("review_timestamp"))
    .withColumn("sentence_id", F.sha2(F.concat_ws("||", "_record_id", F.col("sentence_index").cast("string")), 256))
    .select(
        "sentence_id", F.col("_record_id").alias("record_id"), "id",
        "product_id", "user_id", "score", "sentiment_class",
        "review_time_epoch", "review_timestamp", "review_year",
        "helpfulness_numerator", "helpfulness_denominator", "helpfulness_ratio",
        "sentence_index", "sentence_text", "sentence_normalized",
    )
)
sentence_count = aspect_sentences_df.count()
print(f"Gold aspect-sentence rows: {sentence_count:,}")
aspect_sentences_df.show(10, truncate=100)

## 4. Save and Verify Gold Parquet Outputs

In [ ]:
if model_count == 0 or sentence_count == 0:
    raise ValueError("A Gold output is empty")

model_input_df.write.mode("overwrite").option("compression", "snappy").partitionBy("dataset_split").parquet(GOLD_MODEL_PATH)
aspect_sentences_df.write.mode("overwrite").option("compression", "snappy").parquet(GOLD_ASPECT_PATH)

written_model_count = spark.read.parquet(GOLD_MODEL_PATH).count()
written_sentence_count = spark.read.parquet(GOLD_ASPECT_PATH).count()
if written_model_count != model_count or written_sentence_count != sentence_count:
    raise ValueError("Gold write verification failed")

print(f"Model input: {written_model_count:,} rows -> {GOLD_MODEL_PATH}")
print(f"Aspect sentences: {written_sentence_count:,} rows -> {GOLD_ASPECT_PATH}")
silver_df.unpersist()

In [ ]:
%stop_session